# google_cloud_kms smoke test

Smoke test for `google-cloud-kms`, the client for Google Cloud Key Management
Service. Every example runs offline: it uses the resource-path helpers, builds
key and request messages and round-trips them, and constructs a client with
anonymous credentials. No key is created and no request is sent to Cloud KMS.

## Example 1: Resource path helpers

In [ ]:
from google.cloud import kms

KMS = kms.KeyManagementServiceClient
key_path = KMS.crypto_key_path("p", "global", "ring", "key")
print("crypto key path:", key_path)
assert key_path == "projects/p/locations/global/keyRings/ring/cryptoKeys/key"
assert KMS.parse_crypto_key_path(key_path) == {
    "project": "p", "location": "global", "key_ring": "ring", "crypto_key": "key",
}
assert KMS.key_ring_path("p", "global", "ring") == "projects/p/locations/global/keyRings/ring"

version_path = KMS.crypto_key_version_path("p", "global", "ring", "key", "1")
assert version_path == key_path + "/cryptoKeyVersions/1"
assert KMS.parse_crypto_key_version_path(version_path)["crypto_key_version"] == "1"
print("path helpers OK")

## Example 2: CryptoKey message round trip

In [ ]:
import datetime

from google.cloud import kms

key = kms.CryptoKey(
    purpose=kms.CryptoKey.CryptoKeyPurpose.ENCRYPT_DECRYPT,
    version_template=kms.CryptoKeyVersionTemplate(
        algorithm=kms.CryptoKeyVersion.CryptoKeyVersionAlgorithm.GOOGLE_SYMMETRIC_ENCRYPTION,
        protection_level=kms.ProtectionLevel.SOFTWARE,
    ),
    rotation_period=datetime.timedelta(days=90),
    labels={"env": "demo"},
)
wire = kms.CryptoKey.serialize(key)
back = kms.CryptoKey.deserialize(wire)
print("serialized CryptoKey:", len(wire), "bytes")
assert back == key
assert back.purpose == kms.CryptoKey.CryptoKeyPurpose.ENCRYPT_DECRYPT
assert back.version_template.algorithm.name == "GOOGLE_SYMMETRIC_ENCRYPTION"
assert back.rotation_period == datetime.timedelta(days=90)
assert dict(back.labels) == {"env": "demo"}
assert kms.CryptoKey.to_dict(key)["rotation_period"] == "7776000s"  # protobuf Duration JSON form
print("CryptoKey round trip OK")

## Example 3: Client construction with anonymous credentials

In [ ]:
from google.auth.credentials import AnonymousCredentials
from google.cloud import kms

# Building a client opens no connection; RPCs only start when a method is called.
client = kms.KeyManagementServiceClient(credentials=AnonymousCredentials())
print("transport:", type(client.transport).__name__)
print("endpoint:", client.api_endpoint)
assert client.api_endpoint == "cloudkms.googleapis.com"
assert callable(client.encrypt) and callable(client.decrypt)
print("client constructed")

## Example 4: Encrypt request as JSON

In [ ]:
import base64

from google.cloud import kms

name = kms.KeyManagementServiceClient.crypto_key_path("p", "global", "ring", "key")
request = kms.EncryptRequest(name=name, plaintext=b"hello", additional_authenticated_data=b"ctx")
as_json = kms.EncryptRequest.to_json(request)
print(as_json)
# Bytes fields travel as base64 in the JSON form of a protobuf message.
assert base64.b64encode(b"hello").decode() in as_json
assert kms.EncryptRequest.from_json(as_json) == request
assert kms.EncryptRequest.from_json(as_json).plaintext == b"hello"
print("EncryptRequest JSON round trip OK")

## Example 5: Algorithms, protection levels and digests

In [ ]:
from google.cloud import kms

Algorithm = kms.CryptoKeyVersion.CryptoKeyVersionAlgorithm
assert Algorithm["EC_SIGN_P256_SHA256"] == Algorithm.EC_SIGN_P256_SHA256
assert Algorithm.GOOGLE_SYMMETRIC_ENCRYPTION.name == "GOOGLE_SYMMETRIC_ENCRYPTION"
assert {"SOFTWARE", "HSM", "EXTERNAL"} <= {level.name for level in kms.ProtectionLevel}

# Digest is a oneof: setting sha256 leaves the other digest fields unset.
request = kms.AsymmetricSignRequest(
    name="projects/p/locations/global/keyRings/ring/cryptoKeys/key/cryptoKeyVersions/1",
    digest=kms.Digest(sha256=b"\x00" * 32),
)
assert "sha256" in request.digest and "sha384" not in request.digest
assert len(request.digest.sha256) == 32
print("signing algorithms:", [a.name for a in Algorithm if a.name.startswith("EC_SIGN")])
print("enums and oneof OK")